[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/086.%20The%20Capacitated%20Facility%20Location%20Problem%20%28CFLP%29/P86-Tier-8_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 8: Value-Aligned Decisions and Fairness Metrics

### Purpose and scope

This notebook examines how social, environmental, and stakeholder measures can be used to evaluate facility-location plans alongside cost and service. The example contains eight candidate facilities, ten customer zones, and five stakeholder groups. Its inputs include facility fixed costs and capacities, customer demand and socioeconomic attributes, and facility impact attributes.

The workflow first assigns each zone to the least-cost open facility with enough remaining capacity. It then constructs a small set of plans using four explicit priority strategies (cost, equity, environment, and balance), evaluates those plans, and filters out dominated candidates. This is a heuristic candidate-generation and Pareto-filtering procedure, not an exact solution to the capacitated facility location problem and not a guarantee that the global Pareto frontier has been found.

### Modeling assumptions

- Coordinates are planar, and distance is Euclidean. Transportation cost is 10 cost units per unit of demand per coordinate-distance unit.
- A customer zone is assigned as a whole to one facility. If no open facility has sufficient remaining capacity, that zone's demand is recorded as unmet; partial delivery is not modeled.
- Customer zones are considered in descending demand order for the greedy assignment. Equal-cost ties are resolved by the order in which open facility IDs are iterated.
- The ethical scores are illustrative, dimensionless indicators based on the supplied attributes. They are not empirical measurements or a complete account of social welfare.
- Stakeholder weights for objectives that have no defined score in this example are excluded from that stakeholder's score, and the represented weights are renormalized. Veto power weights the aggregate stakeholder score; it is not a hard feasibility constraint.
- The candidate strategies are deterministic. Repeatedly running a strategy would return the same plan, so the candidate set consists of distinct plans generated by the four strategies.

### Method and interpretation

A plan is represented by a set of open facilities. For a given set, the greedy assignment procedure produces the fixed cost, transportation cost, facility loads, assignments, unmet demand, and service level. A plan's four Pareto objectives are economic efficiency, social equity, environmental sustainability, and stakeholder satisfaction; each is oriented so that larger is preferred. Long-term value is reported as a composite descriptive metric, but is not an additional Pareto objective.

For objectives \(a\) and \(b\), plan \(a\) dominates plan \(b\) if \(a\) is at least as good in every Pareto objective and strictly better in at least one. The nondominated candidates form the Pareto set of the generated candidates only. Several candidates can have identical scores, and ties are retained.

### What to examine

Compare the open facilities and service outcomes produced by the different priorities. Then consider how the score definitions and stakeholder weights influence the resulting ranking. A score is a modeling choice: changing its scale, measurement, or weight can change the assessment, even when the facility plan itself is unchanged.

In [ ]:
from dataclasses import dataclass
from math import isclose, sqrt
from typing import Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
@dataclass
class Facility:
    """A candidate facility and its operational and impact attributes."""
    id: int
    name: str
    x_coord: float
    y_coord: float
    fixed_cost: float
    capacity: float
    community_impact: float
    job_creation: int
    local_tax_revenue: float
    environmental_footprint: float
    accessibility_score: float
    safety_rating: float


@dataclass
class CustomerZone:
    """A demand zone and its socioeconomic attributes."""
    id: int
    name: str
    x_coord: float
    y_coord: float
    demand: float
    median_income: float
    unemployment_rate: float
    population_density: float
    age_demographics: Dict[str, float]
    service_desert: float
    strategic_importance: float


@dataclass
class Stakeholder:
    """A stakeholder group with objective weights and aggregate influence."""
    id: int
    name: str
    type: str
    objectives: List[str]
    weights: Dict[str, float]
    veto_power: float


@dataclass
class EthicalMetrics:
    """Dimensionless indicators used to evaluate a facility plan."""
    economic_efficiency: float
    social_equity: float
    environmental_sustainability: float
    stakeholder_satisfaction: float
    long_term_value: float


@dataclass
class EthicalSolution:
    """A facility plan, its operational evaluation, and its ethical metrics."""
    open_facilities: Set[int]
    total_cost: float
    service_level: float
    ethical_metrics: EthicalMetrics
    stakeholder_evaluations: Dict[int, float]
    pareto_dominated: bool = False


def initialize_facilities() -> List[Facility]:
    """Create the candidate facilities and preserve their supplied attributes."""
    return [
        Facility(1, 'Urban Hub', 25, 75, 200000, 1500, 0.9, 250, 1500000, 0.7, 0.95, 0.9),
        Facility(2, 'Suburban DC', 55, 45, 160000, 1200, 0.6, 180, 800000, 0.5, 0.8, 0.85),
        Facility(3, 'Rural Center', 75, 25, 120000, 800, 0.8, 120, 400000, 0.3, 0.7, 0.8),
        Facility(4, 'Industrial Zone', 85, 65, 180000, 1400, 0.4, 300, 1200000, 0.8, 0.6, 0.75),
        Facility(5, 'Tech Park', 15, 35, 220000, 1000, 0.7, 200, 1800000, 0.4, 0.9, 0.95),
        Facility(6, 'Logistics Hub', 45, 85, 140000, 1100, 0.5, 150, 600000, 0.6, 0.85, 0.8),
        Facility(7, 'Green Facility', 65, 15, 250000, 900, 0.8, 100, 500000, 0.1, 0.95, 0.9),
        Facility(8, 'Community Center', 35, 55, 130000, 700, 0.9, 80, 300000, 0.2, 1.0, 0.85),
    ]


def initialize_customer_zones() -> List[CustomerZone]:
    """Create customer zones, including demand and socioeconomic attributes."""
    return [
        CustomerZone(1, 'Downtown Urban', 30, 70, 600, 75000, 0.05, 5000, {'0-18': 0.2, '19-35': 0.3, '36-65': 0.35, '65+': 0.15}, 0.1, 0.9),
        CustomerZone(2, 'Affluent Suburban', 50, 50, 450, 95000, 0.03, 1500, {'0-18': 0.25, '19-35': 0.25, '36-65': 0.35, '65+': 0.15}, 0.05, 0.8),
        CustomerZone(3, 'Working Class', 70, 30, 380, 45000, 0.08, 2000, {'0-18': 0.3, '19-35': 0.25, '36-65': 0.3, '65+': 0.15}, 0.15, 0.7),
        CustomerZone(4, 'Rural Community', 90, 60, 250, 35000, 0.12, 500, {'0-18': 0.25, '19-35': 0.3, '36-65': 0.3, '65+': 0.15}, 0.4, 0.5),
        CustomerZone(5, 'Industrial Area', 20, 80, 520, 55000, 0.06, 800, {'0-18': 0.2, '19-35': 0.3, '36-65': 0.35, '65+': 0.15}, 0.2, 0.8),
        CustomerZone(6, 'University Town', 60, 40, 420, 60000, 0.04, 3000, {'0-18': 0.15, '19-35': 0.4, '36-65': 0.35, '65+': 0.1}, 0.08, 0.7),
        CustomerZone(7, 'Retirement Community', 80, 20, 280, 40000, 0.03, 400, {'0-18': 0.05, '19-35': 0.05, '36-65': 0.2, '65+': 0.7}, 0.3, 0.4),
        CustomerZone(8, 'Mixed Income', 40, 60, 360, 55000, 0.07, 1200, {'0-18': 0.22, '19-35': 0.28, '36-65': 0.35, '65+': 0.15}, 0.12, 0.6),
        CustomerZone(9, 'Growth Area', 10, 90, 480, 48000, 0.09, 1800, {'0-18': 0.28, '19-35': 0.32, '36-65': 0.3, '65+': 0.1}, 0.18, 0.9),
        CustomerZone(10, 'Established Area', 95, 10, 340, 65000, 0.04, 1000, {'0-18': 0.18, '19-35': 0.22, '36-65': 0.4, '65+': 0.2}, 0.05, 0.5),
    ]


def initialize_stakeholders() -> List[Stakeholder]:
    """Create stakeholder groups with their stated objective weights."""
    return [
        Stakeholder(1, 'Community Coalition', 'COMMUNITY', ['Job Creation', 'Local Economy', 'Environmental Protection', 'Accessibility'], {'job_creation': 0.3, 'local_tax_revenue': 0.2, 'environmental_footprint': 0.3, 'accessibility_score': 0.2}, 0.6),
        Stakeholder(2, 'Employee Union', 'EMPLOYEE', ['Job Creation', 'Safety Rating', 'Fair Wages', 'Working Conditions'], {'job_creation': 0.25, 'safety_rating': 0.4, 'local_tax_revenue': 0.1, 'accessibility_score': 0.25}, 0.4),
        Stakeholder(3, 'Environmental Agency', 'ENVIRONMENT', ['Environmental Protection', 'Sustainability', 'Carbon Footprint', 'Green Spaces'], {'environmental_footprint': 0.4, 'community_impact': 0.2, 'local_tax_revenue': 0.1, 'job_creation': 0.1}, 0.7),
        Stakeholder(4, 'Investors Group', 'INVESTOR', ['Economic Efficiency', 'ROI', 'Market Share', 'Long-term Value'], {'total_cost': 0.4, 'strategic_importance': 0.3, 'expansion_potential': 0.2, 'risk_score': 0.1}, 0.3),
        Stakeholder(5, 'Regulatory Authority', 'REGULATOR', ['Compliance', 'Safety Standards', 'Fair Competition', 'Public Interest'], {'safety_rating': 0.5, 'accessibility_score': 0.3, 'service_level': 0.2, 'community_impact': 0.2}, 0.8),
    ]


facilities = initialize_facilities()
customer_zones = initialize_customer_zones()
stakeholders = initialize_stakeholders()

print('Ethical CFLP example initialized:')
print(f'Facilities: {len(facilities)}')
print(f'Customer zones: {len(customer_zones)}')
print(f'Stakeholder groups: {len(stakeholders)}')

## Assignment, scoring, and Pareto filtering

### Greedy capacity-feasible assignment

For open facility set \(S\), let \(d_j\) be zone \(j\)'s demand, \(K_i\) facility \(i\)'s capacity, and \(c_{ij}\) the transportation cost per demand unit. Zones are processed in descending demand order. Each zone is assigned to the least-cost facility in \(S\) whose remaining capacity can accommodate the entire zone. The code does not solve a separate assignment optimization problem; it applies this specified greedy rule.

The Euclidean distance and unit transportation cost are

\[
\delta_{ij}=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2},\qquad c_{ij}=10\delta_{ij}.
\]

If \(A(S)\) is the set of assigned zones, the reported operating quantities are

\[
C(S)=\sum_{i\in S}F_i+\sum_{j\in A(S)}d_jc_{a(j),j},\qquad
L(S)=\frac{\sum_{j\in A(S)}d_j}{\sum_j d_j},
\]

where \(F_i\) is the fixed cost of facility \(i\), and \(a(j)\) is the facility assigned to zone \(j\). Thus \(L(S)\) is demand-weighted service level. In the implementation, `evaluate_basic_solution` returns these values as well as assignments, facility loads, and unmet demand.

### Ethical indicators

Economic efficiency is normalized against a transparent upper-bound cost: the fixed cost of opening every facility plus the cost of shipping all demand at the largest available per-unit transportation cost. If this bound is \(C_U\), efficiency is \(1-C(S)/C_U\). Social equity averages service equity, coverage balance, job-distribution balance, and tax-distribution balance. Environmental sustainability is one minus the mean environmental footprint of the open facilities. Stakeholder satisfaction is a veto-power-weighted average of stakeholder scores.

The composite long-term value is

\[
V(S)=0.3E(S)+0.3Q_s(S)+0.2Q_e(S)+0.2H(S),
\]

where \(E\) is economic efficiency, \(Q_s\) is service equity, \(Q_e\) is environmental sustainability, and \(H\) is stakeholder satisfaction. It is reported as a summary indicator rather than used in Pareto dominance.

### Candidate strategies and dominance

The generator constructs up to four candidate plans by applying the cost-focused, equity-focused, environmental, and balanced priority rules. For the generated candidate set \(\mathcal{C}\), a plan \(p\) is dominated by \(q\) when

\[
f_k(q)\geq f_k(p)\quad\text{for every Pareto objective }k,
\qquad
f_k(q)>f_k(p)\quad\text{for at least one }k.
\]

The four objectives are economic efficiency, social equity, environmental sustainability, and stakeholder satisfaction. Comparisons use a small numerical tolerance for floating-point values. The filtering stage is quadratic in the number of candidates, \(O(|\mathcal{C}|^2m)\), where \(m=4\) is the number of objectives. Since the strategy set is small, this pairwise check is straightforward; the main limitation is that the resulting set describes only these generated candidates.

In [ ]:
class EthicalCFLOptimizer:
    """Greedy CFLP evaluation with strategy-based candidates and Pareto filtering."""

    def __init__(self, facilities: List[Facility], customer_zones: List[CustomerZone], stakeholders: List[Stakeholder]):
        self.facilities = facilities
        self.customer_zones = customer_zones
        self.stakeholders = stakeholders
        self.facility_by_id = {facility.id: facility for facility in facilities}
        self.zone_by_id = {zone.id: zone for zone in customer_zones}
        self.valid_facility_ids = set(self.facility_by_id)
        self.transport_costs = self._calculate_transport_costs()

    def _calculate_transport_costs(self) -> Dict[Tuple[int, int], float]:
        """Use the stated Euclidean geometry to price each facility-zone pair."""
        costs = {}
        for facility in self.facilities:
            for zone in self.customer_zones:
                distance = sqrt((facility.x_coord - zone.x_coord) ** 2 + (facility.y_coord - zone.y_coord) ** 2)
                costs[(facility.id, zone.id)] = distance * 10.0
        return costs

    def evaluate_basic_solution(self, open_facilities: Set[int]) -> Dict:
        """Greedily assign whole zones subject to open-facility capacities."""
        if not open_facilities <= self.valid_facility_ids:
            raise ValueError('The open-facility set contains an unknown facility ID.')

        fixed_cost = sum(self.facility_by_id[fid].fixed_cost for fid in open_facilities)
        facility_loads = {fid: 0.0 for fid in open_facilities}
        assignments = {}
        transport_cost = 0.0
        unmet_demand = 0.0

        for zone in sorted(self.customer_zones, key=lambda item: item.demand, reverse=True):
            feasible_facilities = [
                fid for fid in open_facilities
                if facility_loads[fid] + zone.demand <= self.facility_by_id[fid].capacity
            ]
            if not feasible_facilities:
                unmet_demand += zone.demand
                continue

            best_facility = min(feasible_facilities, key=lambda fid: self.transport_costs[(fid, zone.id)])
            assignments[zone.id] = best_facility
            facility_loads[best_facility] += zone.demand
            transport_cost += self.transport_costs[(best_facility, zone.id)] * zone.demand

        total_demand = sum(zone.demand for zone in self.customer_zones)
        total_cost = fixed_cost + transport_cost
        service_level = (total_demand - unmet_demand) / total_demand if total_demand else 0.0
        return {
            'total_cost': total_cost,
            'fixed_cost': fixed_cost,
            'transport_cost': transport_cost,
            'service_level': service_level,
            'assignments': assignments,
            'facility_loads': facility_loads,
            'unmet_demand': unmet_demand,
        }

    def _calculate_social_equity(self, open_facilities: Set[int], assignments: Dict[int, int]) -> Dict[str, float]:
        """Calculate the four stated balance indicators on a common score scale."""
        zone_service = []
        for zone in self.customer_zones:
            facility_id = assignments.get(zone.id)
            if facility_id is None:
                zone_service.append(0.0)
            else:
                facility = self.facility_by_id[facility_id]
                zone_service.append((facility.accessibility_score + facility.safety_rating) / 2)

        mean_service = float(np.mean(zone_service)) if zone_service else 0.0
        service_equity = 1.0 - float(np.var(zone_service)) / (mean_service + 0.01)
        service_equity = float(np.clip(service_equity, 0.0, 1.0))

        coverage = {fid: 0 for fid in open_facilities}
        for facility_id in assignments.values():
            coverage[facility_id] += 1
        if coverage:
            max_coverage = max(coverage.values())
            coverage_balance = 1.0 - float(np.std(list(coverage.values()))) / (max_coverage + 0.01)
            coverage_balance = float(np.clip(coverage_balance, 0.0, 1.0))
        else:
            coverage_balance = 0.0

        selected = [self.facility_by_id[fid] for fid in open_facilities]
        if selected:
            jobs = [facility.job_creation for facility in selected]
            taxes = [facility.local_tax_revenue for facility in selected]
            job_distribution = float(np.clip(1.0 - np.std(jobs) / (np.mean(jobs) + 1.0), 0.0, 1.0))
            tax_distribution = float(np.clip(1.0 - np.std(taxes) / (np.mean(taxes) + 1.0), 0.0, 1.0))
        else:
            job_distribution = 0.0
            tax_distribution = 0.0

        return {
            'service_equity': service_equity,
            'coverage_balance': coverage_balance,
            'job_distribution': job_distribution,
            'tax_distribution': tax_distribution,
            'overall_equity': (service_equity + coverage_balance + job_distribution + tax_distribution) / 4.0,
        }

    def _calculate_environmental_impact(self, open_facilities: Set[int]) -> Dict[str, float]:
        """Summarize footprint and community impact for the selected facilities."""
        selected = [self.facility_by_id[fid] for fid in open_facilities]
        if not selected:
            return {'sustainability_score': 0.0, 'avg_footprint': 1.0, 'avg_community_impact': 0.0}
        avg_footprint = float(np.mean([facility.environmental_footprint for facility in selected]))
        avg_community_impact = float(np.mean([facility.community_impact for facility in selected]))
        return {
            'sustainability_score': 1.0 - avg_footprint,
            'avg_footprint': avg_footprint,
            'avg_community_impact': avg_community_impact,
        }

    def calculate_ethical_metrics(self, open_facilities: Set[int]) -> EthicalMetrics:
        """Combine operational results and defined ethical indicators."""
        basic = self.evaluate_basic_solution(open_facilities)
        total_demand = sum(zone.demand for zone in self.customer_zones)
        max_unit_cost = max(self.transport_costs.values(), default=0.0)
        cost_upper_bound = sum(facility.fixed_cost for facility in self.facilities) + total_demand * max_unit_cost
        economic_efficiency = 1.0 - basic['total_cost'] / cost_upper_bound if cost_upper_bound else 0.0
        economic_efficiency = float(np.clip(economic_efficiency, 0.0, 1.0))

        equity = self._calculate_social_equity(open_facilities, basic['assignments'])
        environment = self._calculate_environmental_impact(open_facilities)
        stakeholder_satisfaction = self._calculate_stakeholder_satisfaction(
            open_facilities, basic, equity, environment, economic_efficiency
        )
        long_term_value = (
            0.3 * economic_efficiency
            + 0.3 * equity['service_equity']
            + 0.2 * environment['sustainability_score']
            + 0.2 * stakeholder_satisfaction
        )
        return EthicalMetrics(
            economic_efficiency=economic_efficiency,
            social_equity=equity['overall_equity'],
            environmental_sustainability=environment['sustainability_score'],
            stakeholder_satisfaction=stakeholder_satisfaction,
            long_term_value=long_term_value,
        )

    def _calculate_stakeholder_satisfaction(
        self,
        open_facilities: Set[int],
        basic_solution: Dict,
        equity_metrics: Dict[str, float],
        environmental_metrics: Dict[str, float],
        economic_efficiency: float,
    ) -> float:
        """Aggregate only stakeholder objectives for which this example defines scores."""
        selected = [self.facility_by_id[fid] for fid in open_facilities]
        all_jobs = sum(facility.job_creation for facility in self.facilities)
        all_tax = sum(facility.local_tax_revenue for facility in self.facilities)
        mean_accessibility = float(np.mean([facility.accessibility_score for facility in selected])) if selected else 0.0
        mean_safety = float(np.mean([facility.safety_rating for facility in selected])) if selected else 0.0
        mean_community = float(np.mean([facility.community_impact for facility in selected])) if selected else 0.0
        selected_jobs = sum(facility.job_creation for facility in selected)
        selected_tax = sum(facility.local_tax_revenue for facility in selected)

        scores = {
            'total_cost': economic_efficiency,
            'service_level': basic_solution['service_level'],
            'service_equity': equity_metrics['service_equity'],
            'coverage_balance': equity_metrics['coverage_balance'],
            'environmental_footprint': environmental_metrics['sustainability_score'],
            'job_creation': selected_jobs / all_jobs if all_jobs else 0.0,
            'local_tax_revenue': selected_tax / all_tax if all_tax else 0.0,
            'accessibility_score': mean_accessibility,
            'safety_rating': mean_safety,
            'community_impact': mean_community,
        }

        weighted_satisfaction = 0.0
        total_veto_weight = 0.0
        for stakeholder in self.stakeholders:
            represented = [(scores[key], weight) for key, weight in stakeholder.weights.items() if key in scores]
            represented_weight = sum(weight for _, weight in represented)
            if represented_weight <= 0.0:
                continue
            score = sum(value * weight for value, weight in represented) / represented_weight
            weighted_satisfaction += stakeholder.veto_power * score
            total_veto_weight += stakeholder.veto_power
        return weighted_satisfaction / total_veto_weight if total_veto_weight else 0.0

    def _make_solution(self, open_facilities: Set[int]) -> EthicalSolution:
        evaluation = self.evaluate_basic_solution(open_facilities)
        metrics = self.calculate_ethical_metrics(open_facilities)
        return EthicalSolution(set(open_facilities), evaluation['total_cost'], evaluation['service_level'], metrics, {})

    def _generate_cost_optimized_solution(self) -> Optional[EthicalSolution]:
        """Open facilities in ascending fixed-cost order until service reaches 90%."""
        open_facilities: Set[int] = set()
        evaluation = self.evaluate_basic_solution(open_facilities)
        for facility in sorted(self.facilities, key=lambda item: item.fixed_cost):
            if evaluation['service_level'] >= 0.9:
                break
            open_facilities.add(facility.id)
            evaluation = self.evaluate_basic_solution(open_facilities)
        return self._make_solution(open_facilities) if open_facilities else None

    def _generate_equity_focused_solution(self) -> Optional[EthicalSolution]:
        """Prioritize zones with greater service-desert, income, and strategic weights."""
        priorities = {
            zone.id: 0.4 * zone.service_desert + 0.3 * (1.0 - zone.median_income / 100000.0)
            + 0.3 * zone.strategic_importance
            for zone in self.customer_zones
        }
        open_facilities: Set[int] = set()
        for zone in sorted(self.customer_zones, key=lambda item: priorities[item.id], reverse=True):
            candidates = [facility for facility in self.facilities if facility.id not in open_facilities]
            if not candidates:
                break
            nearest = min(candidates, key=lambda facility: self.transport_costs[(facility.id, zone.id)])
            open_facilities.add(nearest.id)
        return self._make_solution(open_facilities) if open_facilities else None

    def _generate_environmental_solution(self) -> Optional[EthicalSolution]:
        """Open low-footprint facilities first and add capacity if service requires it."""
        green = sorted(
            [facility for facility in self.facilities if facility.environmental_footprint < 0.5],
            key=lambda item: item.environmental_footprint,
        )
        open_facilities = {facility.id for facility in green}
        evaluation = self.evaluate_basic_solution(open_facilities)
        if evaluation['service_level'] < 0.85:
            others = sorted(
                [facility for facility in self.facilities if facility.id not in open_facilities],
                key=lambda item: item.fixed_cost,
            )
            for facility in others:
                open_facilities.add(facility.id)
                evaluation = self.evaluate_basic_solution(open_facilities)
                if evaluation['service_level'] >= 0.85:
                    break
        return self._make_solution(open_facilities) if open_facilities else None

    def _generate_balanced_solution(self) -> Optional[EthicalSolution]:
        """Rank facilities with equal weights on cost, community, and footprint scores."""
        max_fixed_cost = max(facility.fixed_cost for facility in self.facilities)
        facility_scores = []
        for facility in self.facilities:
            cost_score = 1.0 - facility.fixed_cost / max_fixed_cost
            environmental_score = 1.0 - facility.environmental_footprint
            score = 0.25 * cost_score + 0.25 * facility.community_impact + 0.25 * facility.community_impact + 0.25 * environmental_score
            facility_scores.append((facility.id, score))

        open_facilities: Set[int] = set()
        evaluation = self.evaluate_basic_solution(open_facilities)
        for facility_id, _ in sorted(facility_scores, key=lambda item: item[1], reverse=True):
            if evaluation['service_level'] >= 0.9:
                break
            open_facilities.add(facility_id)
            evaluation = self.evaluate_basic_solution(open_facilities)
        return self._make_solution(open_facilities) if open_facilities else None

    def _dominates(self, solution1: EthicalSolution, solution2: EthicalSolution) -> bool:
        """Test Pareto dominance across the four maximized objectives."""
        values1 = (
            solution1.ethical_metrics.economic_efficiency,
            solution1.ethical_metrics.social_equity,
            solution1.ethical_metrics.environmental_sustainability,
            solution1.ethical_metrics.stakeholder_satisfaction,
        )
        values2 = (
            solution2.ethical_metrics.economic_efficiency,
            solution2.ethical_metrics.social_equity,
            solution2.ethical_metrics.environmental_sustainability,
            solution2.ethical_metrics.stakeholder_satisfaction,
        )
        no_worse = all(a > b or isclose(a, b, rel_tol=1e-9, abs_tol=1e-12) for a, b in zip(values1, values2))
        strictly_better = any(a > b and not isclose(a, b, rel_tol=1e-9, abs_tol=1e-12) for a, b in zip(values1, values2))
        return no_worse and strictly_better

    def generate_pareto_solutions(self) -> List[EthicalSolution]:
        """Generate one candidate per priority strategy and retain nondominated plans."""
        strategies = (
            self._generate_cost_optimized_solution,
            self._generate_equity_focused_solution,
            self._generate_environmental_solution,
            self._generate_balanced_solution,
        )
        candidates = []
        seen_facility_sets = set()
        for strategy in strategies:
            solution = strategy()
            if solution is not None:
                key = frozenset(solution.open_facilities)
                if key not in seen_facility_sets:
                    candidates.append(solution)
                    seen_facility_sets.add(key)

        for candidate in candidates:
            candidate.pareto_dominated = any(
                other is not candidate and self._dominates(other, candidate)
                for other in candidates
            )
        return [candidate for candidate in candidates if not candidate.pareto_dominated]

In [ ]:
optimizer = EthicalCFLOptimizer(facilities, customer_zones, stakeholders)
pareto_solutions = optimizer.generate_pareto_solutions()

if not pareto_solutions:
    raise RuntimeError('The candidate strategies did not produce a nondominated plan.')

print('=' * 72)
print('NONDOMINATED STRATEGY CANDIDATES')
print('=' * 72)
print(f'Number of distinct nondominated candidates: {len(pareto_solutions)}')
print('\nCandidate metrics:')
for index, solution in enumerate(pareto_solutions, start=1):
    metrics = solution.ethical_metrics
    print(f'\nCandidate {index}: open facilities {sorted(solution.open_facilities)}')
    print(f'  Total cost: ${solution.total_cost:,.2f}')
    print(f'  Demand-weighted service level: {solution.service_level:.1%}')
    print(f'  Economic efficiency: {metrics.economic_efficiency:.3f}')
    print(f'  Social equity: {metrics.social_equity:.3f}')
    print(f'  Environmental sustainability: {metrics.environmental_sustainability:.3f}')
    print(f'  Stakeholder satisfaction: {metrics.stakeholder_satisfaction:.3f}')
    print(f'  Long-term value index: {metrics.long_term_value:.3f}')

cost_optimal = min(pareto_solutions, key=lambda solution: solution.total_cost)
service_optimal = max(pareto_solutions, key=lambda solution: solution.service_level)
equity_optimal = max(pareto_solutions, key=lambda solution: solution.ethical_metrics.social_equity)
environmental_optimal = max(
    pareto_solutions,
    key=lambda solution: solution.ethical_metrics.environmental_sustainability,
)
evaluation_solutions = [cost_optimal, service_optimal, equity_optimal, environmental_optimal]

costs = [solution.total_cost for solution in pareto_solutions]
service_levels = [solution.service_level for solution in pareto_solutions]
equity_scores = [solution.ethical_metrics.social_equity for solution in pareto_solutions]
environmental_scores = [solution.ethical_metrics.environmental_sustainability for solution in pareto_solutions]
print('\nRanges across the generated nondominated candidates:')
print(f'  Cost: ${min(costs):,.2f} to ${max(costs):,.2f}')
print(f'  Service level: {min(service_levels):.1%} to {max(service_levels):.1%}')
print(f'  Social equity: {min(equity_scores):.3f} to {max(equity_scores):.3f}')
print(f'  Environmental sustainability: {min(environmental_scores):.3f} to {max(environmental_scores):.3f}')
print('\nRepresentative candidates are selected by the indicated single metric; ties may occur.')

## Reading the candidate set

The reported ranges summarize only the distinct nondominated plans produced by the four priority strategies. They are not bounds over all feasible CFLP plans. A cost-optimal candidate is the least-cost member of this generated set, not necessarily the globally least-cost facility plan. Likewise, a nondominated candidate is not automatically preferable: Pareto filtering identifies trade-offs but does not choose the values that a decision maker should prioritize.

When a metric is not different among the generated candidates, that indicates only that these strategies did not create a visible trade-off on that metric. It does not show that the metric is irrelevant in the full facility-location problem. Review the underlying assignments and unmet demand as well as the summary scores.

In [ ]:
def evaluate_stakeholders(solution: EthicalSolution) -> Dict[int, float]:
    """Score each stakeholder on the objectives represented by the model."""
    basic = optimizer.evaluate_basic_solution(solution.open_facilities)
    equity = optimizer._calculate_social_equity(solution.open_facilities, basic['assignments'])
    environment = optimizer._calculate_environmental_impact(solution.open_facilities)
    metrics = solution.ethical_metrics
    selected = [optimizer.facility_by_id[fid] for fid in solution.open_facilities]
    all_jobs = sum(facility.job_creation for facility in facilities)
    all_tax = sum(facility.local_tax_revenue for facility in facilities)
    scores = {
        'total_cost': metrics.economic_efficiency,
        'service_level': solution.service_level,
        'service_equity': equity['service_equity'],
        'coverage_balance': equity['coverage_balance'],
        'environmental_footprint': environment['sustainability_score'],
        'job_creation': sum(facility.job_creation for facility in selected) / all_jobs if all_jobs else 0.0,
        'local_tax_revenue': sum(facility.local_tax_revenue for facility in selected) / all_tax if all_tax else 0.0,
        'accessibility_score': float(np.mean([facility.accessibility_score for facility in selected])) if selected else 0.0,
        'safety_rating': float(np.mean([facility.safety_rating for facility in selected])) if selected else 0.0,
        'community_impact': float(np.mean([facility.community_impact for facility in selected])) if selected else 0.0,
    }
    evaluations = {}
    for stakeholder in stakeholders:
        represented = [(scores[key], weight) for key, weight in stakeholder.weights.items() if key in scores]
        total_weight = sum(weight for _, weight in represented)
        score = sum(value * weight for value, weight in represented) / total_weight if total_weight else 0.0
        evaluations[stakeholder.id] = score * stakeholder.veto_power
    return evaluations

for solution in evaluation_solutions:
    solution.stakeholder_evaluations = evaluate_stakeholders(solution)

print('=' * 72)
print('STAKEHOLDER EVALUATION OF REPRESENTATIVE CANDIDATES')
print('=' * 72)
for index, solution in enumerate(evaluation_solutions, start=1):
    print(f'\nRepresentative candidate {index}: facilities {sorted(solution.open_facilities)}')
    for stakeholder in stakeholders:
        print(f'  {stakeholder.name}: {solution.stakeholder_evaluations[stakeholder.id]:.3f}')

# This example uses the Environmental Agency's weighted score as a transparent selection rule.
environmental_agency = next(stakeholder for stakeholder in stakeholders if stakeholder.id == 3)
best_solution = max(
    evaluation_solutions,
    key=lambda solution: solution.stakeholder_evaluations[environmental_agency.id],
)
overall_satisfaction = best_solution.ethical_metrics.stakeholder_satisfaction
print('\nSelected representative candidate under the Environmental Agency score:')
print(f'  Cost: ${best_solution.total_cost:,.2f}')
print(f'  Service level: {best_solution.service_level:.1%}')
print(f'  Open facilities: {sorted(best_solution.open_facilities)}')
print(f'  Facility names: {", ".join(optimizer.facility_by_id[fid].name for fid in sorted(best_solution.open_facilities))}')

### Stakeholder scores and decision authority

Each stakeholder score is a weighted average of the objectives for which this example defines a score, multiplied by that stakeholder's veto-power value. Objectives such as fair wages, risk, and expansion potential have no corresponding data or score here; they are not silently treated as measured outcomes. The final representative plan is selected by maximizing the Environmental Agency's stakeholder score among the four metric-based representatives. This is a stated decision rule for the illustration, not a mathematical veto or a claim that the selected plan represents unanimous stakeholder agreement.

The four representative labels (cost, service, equity, and environment) identify the metric used to select a plan from the generated candidate set. If multiple candidates tie on that metric, Python's `min` or `max` returns one of them according to iteration order; a tie does not imply a unique best plan.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('CFLP Candidate Plans: Cost, Service, and Ethical Indicators', fontsize=16, fontweight='bold')

# Cost and service are shown directly; color encodes the social-equity score.
ax = axes[0, 0]
scatter = ax.scatter(
    [solution.total_cost for solution in pareto_solutions],
    [solution.service_level for solution in pareto_solutions],
    c=[solution.ethical_metrics.social_equity for solution in pareto_solutions],
    s=100,
    cmap='viridis',
    edgecolors='black',
)
fig.colorbar(scatter, ax=ax, label='Social equity score')
ax.set_xlabel('Total cost (cost units)')
ax.set_ylabel('Demand-weighted service level')
ax.set_title('Nondominated Strategy Candidates')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# Compare each representative plan on the four maximized Pareto objectives.
ax = axes[0, 1]
metric_names = ['Economic efficiency', 'Social equity', 'Environmental sustainability', 'Stakeholder satisfaction']
metric_values = [
    [solution.ethical_metrics.economic_efficiency for solution in evaluation_solutions],
    [solution.ethical_metrics.social_equity for solution in evaluation_solutions],
    [solution.ethical_metrics.environmental_sustainability for solution in evaluation_solutions],
    [solution.ethical_metrics.stakeholder_satisfaction for solution in evaluation_solutions],
]
x = np.arange(len(metric_names))
width = 0.19
for index, solution in enumerate(evaluation_solutions):
    values = [series[index] for series in metric_values]
    ax.bar(x + (index - 1.5) * width, values, width, label=f'Representative {index + 1}')
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=25, ha='right')
ax.set_ylabel('Score (higher is preferred)')
ax.set_title('Representative Plan Metrics')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# The matrix describes similarity in the four Pareto-objective scores.
ax = axes[0, 2]
solution_names = ['Cost', 'Service', 'Equity', 'Environment']
comparison_matrix = np.zeros((len(evaluation_solutions), len(evaluation_solutions)))
for i, solution1 in enumerate(evaluation_solutions):
    values1 = np.array([
        solution1.ethical_metrics.economic_efficiency,
        solution1.ethical_metrics.social_equity,
        solution1.ethical_metrics.environmental_sustainability,
        solution1.ethical_metrics.stakeholder_satisfaction,
    ])
    for j, solution2 in enumerate(evaluation_solutions):
        values2 = np.array([
            solution2.ethical_metrics.economic_efficiency,
            solution2.ethical_metrics.social_equity,
            solution2.ethical_metrics.environmental_sustainability,
            solution2.ethical_metrics.stakeholder_satisfaction,
        ])
        comparison_matrix[i, j] = 1.0 - float(np.mean(np.abs(values1 - values2)))
sns.heatmap(comparison_matrix, annot=True, fmt='.2f', cmap='RdYlBu', vmin=0, vmax=1, ax=ax)
ax.set_xticklabels(solution_names, rotation=30, ha='right')
ax.set_yticklabels(solution_names, rotation=0)
ax.set_title('Similarity of Pareto-Objective Scores')

# Illustrate the chosen facilities and all customer-zone locations.
ax = axes[1, 0]
for facility in facilities:
    is_open = facility.id in best_solution.open_facilities
    ax.scatter(
        facility.x_coord,
        facility.y_coord,
        color='red' if is_open else 'lightgray',
        marker='s',
        s=180 if is_open else 100,
        edgecolors='black',
        label='Open facility' if is_open and facility.id == min(best_solution.open_facilities) else ('Closed facility' if not is_open and facility.id == min(set(f.id for f in facilities) - best_solution.open_facilities) else None),
    )
    ax.annotate(f'F{facility.id}', (facility.x_coord, facility.y_coord), xytext=(4, 4), textcoords='offset points', fontsize=8)
for zone in customer_zones:
    ax.scatter(zone.x_coord, zone.y_coord, color='blue', marker='^', s=65, edgecolors='black', label='Customer zone' if zone.id == customer_zones[0].id else None)
    ax.annotate(f'Z{zone.id}', (zone.x_coord, zone.y_coord), xytext=(4, -9), textcoords='offset points', fontsize=7)
ax.set_xlabel('X coordinate (distance units)')
ax.set_ylabel('Y coordinate (distance units)')
ax.set_title('Selected Plan and Customer Zones')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Show which representative plan was chosen and its reported composite index.
ax = axes[1, 1]
representative_values = [solution.ethical_metrics.long_term_value for solution in evaluation_solutions]
ax.bar(solution_names, representative_values, color=['steelblue', 'darkorange', 'seagreen', 'mediumpurple'])
ax.set_ylabel('Long-term value index')
ax.set_ylim(0, 1.05)
ax.set_title('Composite Long-Term Value Indicator')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)

# Show the service and unmet-demand consequences of the selected plan.
ax = axes[1, 2]
selected_evaluation = optimizer.evaluate_basic_solution(best_solution.open_facilities)
assigned_demand = sum(optimizer.zone_by_id[zone_id].demand for zone_id in selected_evaluation['assignments'])
unmet_demand = selected_evaluation['unmet_demand']
ax.bar(['Assigned demand', 'Unmet demand'], [assigned_demand, unmet_demand], color=['seagreen', 'firebrick'])
ax.set_ylabel('Demand units')
ax.set_title('Demand Outcome for Selected Plan')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Interpreting the figures

The scatter plot shows the generated candidates in cost-service space, with color indicating social equity. The other panels compare the representative plans using defined scores, show score similarity, locate the selected facilities and customer zones, and display the selected plan's service outcome. Geographic positions are illustrative coordinates; no road network, travel time, or geographic boundary data are represented.

Similarity in the heatmap is \(1\) minus the mean absolute difference across the four Pareto-objective scores. It measures closeness in those normalized scores, not similarity of facility locations or stakeholder opinions. The long-term value panel displays the composite index defined above; it is not a forecast of future monetary or societal outcomes.

In [ ]:
print('=' * 72)
print('FINAL PLAN SUMMARY')
print('=' * 72)
print(f'Total cost: ${best_solution.total_cost:,.2f}')
print(f'Demand-weighted service level: {best_solution.service_level:.1%}')
print(f'Open facilities: {len(best_solution.open_facilities)}')
print(f'Facility IDs: {sorted(best_solution.open_facilities)}')
print('\nEthical indicators:')
print(f'  Economic efficiency: {best_solution.ethical_metrics.economic_efficiency:.3f}')
print(f'  Social equity: {best_solution.ethical_metrics.social_equity:.3f}')
print(f'  Environmental sustainability: {best_solution.ethical_metrics.environmental_sustainability:.3f}')
print(f'  Stakeholder satisfaction: {best_solution.ethical_metrics.stakeholder_satisfaction:.3f}')
print(f'  Long-term value index: {best_solution.ethical_metrics.long_term_value:.3f}')

baseline_cost = cost_optimal.total_cost
cost_premium = (best_solution.total_cost - baseline_cost) / baseline_cost * 100 if baseline_cost else 0.0
print(f'\nCost difference relative to the least-cost generated candidate: {cost_premium:+.1f}%')
print('Stakeholder scores for the selected plan:')
for stakeholder in stakeholders:
    print(f'  {stakeholder.name}: {best_solution.stakeholder_evaluations[stakeholder.id]:.3f}')

selected_facilities = [facility for facility in facilities if facility.id in best_solution.open_facilities]
selected_jobs = sum(facility.job_creation for facility in selected_facilities)
selected_tax = sum(facility.local_tax_revenue for facility in selected_facilities)
selected_evaluation = optimizer.evaluate_basic_solution(best_solution.open_facilities)
served_desert_zones = sum(
    1 for zone in customer_zones
    if zone.service_desert > 0.2 and zone.id in selected_evaluation['assignments']
)
selected_footprint = float(np.mean([facility.environmental_footprint for facility in selected_facilities])) if selected_facilities else 0.0
all_facility_footprint = float(np.mean([facility.environmental_footprint for facility in facilities]))
footprint_change = (all_facility_footprint - selected_footprint) / all_facility_footprint * 100 if all_facility_footprint else 0.0
print('\nDescriptive outcomes for the selected facilities:')
print(f'  Jobs represented by selected facilities: {selected_jobs}')
print(f'  Local tax revenue represented by selected facilities: ${selected_tax:,.2f}')
print(f'  Served zones with service-desert indicator above 0.2: {served_desert_zones}')
print(f'  Footprint change relative to the mean across all candidate facilities: {footprint_change:+.1f}%')
print(f'  Unmet demand: {selected_evaluation["unmet_demand"]:,.1f} demand units')

# These checks connect the displayed plan to the assignment and score calculations.
assert best_solution.open_facilities <= optimizer.valid_facility_ids
assert isclose(best_solution.total_cost, selected_evaluation['fixed_cost'] + selected_evaluation['transport_cost'], rel_tol=1e-9, abs_tol=1e-6)
assert isclose(best_solution.service_level, selected_evaluation['service_level'], rel_tol=1e-9, abs_tol=1e-12)
assert 0.0 <= best_solution.service_level <= 1.0
assert 0.0 <= best_solution.ethical_metrics.economic_efficiency <= 1.0
print('\nValidation checks passed for the selected plan.')

## Strengths, limitations, and appropriate use

### What this framework demonstrates

- Operational cost and capacity can be evaluated together with explicitly defined social and environmental indicators.
- A Pareto comparison can preserve nondominated alternatives instead of hiding every trade-off inside one weighted objective.
- Stakeholder preferences can be made visible through documented weights and a transparent selection rule.
- A plan can be audited through its facility set, assignments, facility loads, service level, and score components.

### Limitations

- The greedy, whole-zone assignment rule is order-dependent and is not guaranteed to minimize transportation cost or maximize service for a fixed open-facility set.
- The strategy generator examines only four priority rules. Its nondominated set is not the complete Pareto frontier, and the selected plan is not a globally optimal CFLP solution.
- Several stakeholder objectives in the input have no operational measure in this example. The resulting stakeholder scores therefore represent only the objectives for which a score is defined.
- The social-equity and environmental indicators are simplified proxies. Their scales and definitions require substantive review before they could support a real policy or investment decision.
- The model omits such factors as road-network travel, emissions by shipment, facility-opening constraints, workforce conditions, uncertainty, and stakeholder negotiation.

### When to use this style of analysis

It is useful for teaching how values and stakeholder perspectives can be represented explicitly in a facility-location analysis, and for organizing discussion of trade-offs among a small number of candidate plans. Any real application would require validated data, stakeholder-approved metrics and weights, and a careful review of the operational and ethical assumptions. Natural extensions include adding justified measures or constraints and evaluating larger candidate sets while retaining a clear account of how each plan is produced.